# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nLicense: {metadata.license}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their @id's.

- Each record set, field, and column is uniquely referenced by its `@id`.
- Let's list available record sets with their `@id`s, names, and fields.

In [ ]:
# List all record sets, fields, and columns with their @id
record_sets = dataset.metadata.recordSet
if record_sets:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name','')}")
        fields = rs.get('field', [])
        for f in fields:
            print(f"  Field @id: {f['@id']}, name: {f.get('name','')}, dataType: {f.get('dataType','')} ")
        print()
else:
    print("No record sets found in metadata. The schema may need to be explored directly.")

# Example: List records for a record set by its @id
# (Replace <record_set_id> with actual @id as found above)
# for x in dataset.records(record_set='<record_set_id>'):
#     print(x)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

- Use the record set and field `@id`s from the overview.

In [ ]:
# We assume the primary survey data is stored in a record set with @id:
main_record_set_id = 'https://api.app.sen.science/frontiers/7160411/mental-health-survey-recordset'
# Use actual @id found from previous cell (replace if schema varies)

record_sets_ids = []
if record_sets:
    record_sets_ids = [rs['@id'] for rs in record_sets]
else:
    # If no record sets in metadata, specify manually
    record_sets_ids = [main_record_set_id]

dataframes = {}

for record_set in record_sets_ids:
    records = list(dataset.records(record_set=record_set))
    if records:
        dataframes[record_set] = pd.DataFrame(records)

# Show columns and preview for the main record set
if main_record_set_id in dataframes:
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
elif dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- Removing outliers
- Transforming fields
- Grouping data

We'll pick one of the numeric fields, such as GAD-7 score, for demonstration.
All fields and columns should be referenced by their `@id`.

In [ ]:
# Suppose GAD-7 score is stored in column with @id:
gad7_field_id = 'https://api.app.sen.science/frontiers/7160411/gad7_score'

record_set_id = main_record_set_id if main_record_set_id in dataframes else list(dataframes.keys())[0]
df = dataframes[record_set_id]

numeric_field = gad7_field_id

# If columns are not mapped with @id, match column name instead:
if numeric_field not in df.columns:
    # Try matching by substring or known label
    found_cols = [col for col in df.columns if 'gad7' in col.lower() or 'score' in col.lower()]
    if found_cols:
        numeric_field = found_cols[0]

threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by demographic variable, e.g. age group. Suppose @id:
group_field_id = 'https://api.app.sen.science/frontiers/7160411/age_group'
group_field = group_field_id

if group_field not in df.columns:
    # Try matching by substring or known label
    found_group_cols = [col for col in df.columns if 'age' in col.lower() and 'group' in col.lower()]
    if found_group_cols:
        group_field = found_group_cols[0]

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- Distribution of GAD-7 scores
- GAD-7 scores by age group
- All references use `@id` where possible

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# GAD-7 score distribution
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f'Distribution of GAD-7 Scores ({numeric_field})')
plt.xlabel('GAD-7 Score')
plt.ylabel('Frequency')
plt.show()

# GAD-7 score by Age Group
if group_field in df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f'GAD-7 Scores by Age Group ({group_field})')
    plt.xlabel('Age Group')
    plt.ylabel('GAD-7 Score')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset provides structured mental health survey data including demographic and clinical variables, uniquely referenced by their `@id`.
- Data loading and access with `mlcroissant` enables direct mapping to schema entities and fields.
- GAD-7 score distributions reveal variability and potential cases for further public health intervention in Kilifi County.
- Schema-based referencing facilitates reproducibility and clarity for AI-ready health datasets.

Further extensions: compare additional variables (PHQ-9, PSQ scores), perform correlation analysis, and evaluate subgroup patterns for health insights.